In [20]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.catboost
import os
import json
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from scipy.stats import skew

os.environ['MLFLOW_TRACKING_USERNAME'] = 'ejoba22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'a7173ed993c5e34d5a5b3661dc4380bb976e814f'

mlflow.set_tracking_uri("https://dagshub.com/ejoba22/House-Prices.mlflow")
print("Connected!")

Connected!


In [21]:
model = mlflow.catboost.load_model("models:/catboost_baseline/3")
print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'catboost.core.CatBoostRegressor'>


In [22]:
#Load test data & saved artifacts 
test = pd.read_csv('house-prices-advanced-regression-techniques/test.csv')
test_ids = test['Id'].copy()
test.drop(['Id'], axis=1, inplace=True)

skewed_features   = json.load(open('skewed_features.json'))
train_columns     = json.load(open('train_columns.json'))
lot_medians       = json.load(open('lot_frontage_medians.json'))
garage_medians    = json.load(open('garage_area_medians.json'))

print(f"Test shape: {test.shape}")
print(f"Skewed features to transform: {len(skewed_features)}")
print(f"Expected columns after processing: {len(train_columns)}")

Test shape: (1459, 79)
Skewed features to transform: 47
Expected columns after processing: 238


In [23]:
#NaN Filling (must exactly match experiment notebook) 
null_with_meaning = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual",
    "GarageCond", "PoolQC", "Fence", "MiscFeature"
]
for col in null_with_meaning:
    test[col] = test[col].fillna("None")


features_modefill = [
    "MSZoning", "Utilities", "Exterior1st", "Exterior2nd",
    "SaleType", "Electrical", "KitchenQual", "Functional", "MasVnrType"
]
for col in features_modefill:
    if col in test.columns:
        test[col] = test[col].fillna(test[col].mode()[0])

overall_lot_median     = np.median(list(lot_medians.values()))
overall_garage_median  = np.median(list(garage_medians.values()))

test["LotFrontage"] = test.apply(
    lambda r: lot_medians.get(r["Neighborhood"], overall_lot_median)
    if pd.isna(r["LotFrontage"]) else r["LotFrontage"],
    axis=1
)
test["GarageArea"] = test.apply(
    lambda r: garage_medians.get(r["Neighborhood"], overall_garage_median)
    if pd.isna(r["GarageArea"]) else r["GarageArea"],
    axis=1
)

features_zerofill = [
    "GarageYrBlt", "MasVnrArea", "BsmtHalfBath",
    "BsmtFullBath", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "GarageCars"
]
for col in features_zerofill:
    test[col] = test[col].fillna(0)

remaining = test[features_zerofill + ["LotFrontage", "GarageArea"]].isnull().sum().sum()
print(f"NaNs remaining after fill: {remaining}")  

NaNs remaining after fill: 0


In [24]:
#Feature Engineering 

test["TotalArea"]      = test["GrLivArea"] + test["TotalBsmtSF"]
test["TotalBaths"]     = (test["FullBath"] + test["BsmtFullBath"]
                          + 0.5 * (test["HalfBath"] + test["BsmtHalfBath"]))
test["TotalPorch"]     = (test["OpenPorchSF"] + test["EnclosedPorch"]
                          + test["3SsnPorch"] + test["ScreenPorch"])

test["HouseAge"]       = (test["YrSold"] - test["YearBuilt"]).clip(lower=0)
test["RemodAge"]       = (test["YrSold"] - test["YearRemodAdd"]).clip(lower=0)

test["OverallScore"]   = test["OverallQual"] * test["OverallCond"]
test["LivArea_Qual"]   = test["GrLivArea"] * test["OverallQual"]
test["LivArea_Age"]    = test["GrLivArea"] / (test["HouseAge"] + 1)
test["Qual_TotalArea"] = test["OverallQual"] * test["TotalArea"]

test['Pool']      = (test['PoolArea'] > 0).astype(int)
test['2ndFloor']  = (test['2ndFlrSF'] > 0).astype(int)
test['Garage']    = (test['GarageCars'] > 0).astype(int)
test['Bsmt']      = (test['TotalBsmtSF'] > 0).astype(int)
test['Fireplace'] = (test['Fireplaces'] > 0).astype(int)
test['Porch']     = (test['TotalPorch'] > 0).astype(int)

test["MoSoldsin"] = np.sin(2 * np.pi * test["MoSold"] / 12)
test["MoSoldcos"] = np.cos(2 * np.pi * test["MoSold"] / 12)
test.drop("MoSold", axis=1, inplace=True)

print(f"Shape after feature engineering: {test.shape}")

Shape after feature engineering: (1459, 95)


In [25]:
# Encoding
quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ordinal_cols = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
    "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"
]
for col in ordinal_cols:
    test[col] = test[col].map(quality_map).fillna(0).astype(int)

other_ordinal = {
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Fence":        {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Functional":   {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "LandSlope":    {"Sev": 1, "Mod": 2, "Gtl": 3},
    "LotShape":     {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "Street":       {"Grvl": 0, "Pave": 1},
    "CentralAir":   {"N": 0, "Y": 1},
    "Utilities":    {"ELO": 1, "NoSeWa": 2, "NoSewr": 3, "AllPub": 4},
}
for col, mapping in other_ordinal.items():
    if col in test.columns:
        test[col] = test[col].map(mapping).fillna(0).astype(int)

# One-hot encode nominal columns
nominal_cols = [c for c in test.select_dtypes(include="object").columns]
test = pd.get_dummies(test, columns=nominal_cols)

print(f"Shape after encoding: {test.shape}")

Shape after encoding: (1459, 223)


In [26]:
test = test.reindex(columns=train_columns, fill_value=0)

print(f"Shape after column alignment: {test.shape}")
print(f"Columns match training: {test.shape[1] == len(train_columns)}")

Shape after column alignment: (1459, 238)
Columns match training: True


In [27]:
for feat in skewed_features:
    if feat in test.columns:
        test[feat] = np.log1p(test[feat].clip(lower=0))

inf_count = np.isinf(test.select_dtypes(include=[np.number]).values).sum()
nan_count = test.isnull().sum().sum()
print(f"Inf values: {inf_count} | NaN values: {nan_count}")  # both should be 0

Inf values: 0 | NaN values: 0


In [28]:
model_features = model.feature_names_
missing_in_test = set(model_features) - set(test.columns)
extra_in_test   = set(test.columns) - set(model_features)

print(f"Features expected by model: {len(model_features)}")
print(f"Missing from test (will be filled with 0): {len(missing_in_test)}")
print(f"Extra in test (will be dropped): {len(extra_in_test)}")

if missing_in_test:
    print("Missing:", list(missing_in_test)[:10])
if extra_in_test:
    print("Extra:", list(extra_in_test)[:10])

test = test.reindex(columns=model_features, fill_value=0)
print(f"\nFinal test shape: {test.shape}")

Features expected by model: 238
Missing from test (will be filled with 0): 0
Extra in test (will be dropped): 0

Final test shape: (1459, 238)


In [29]:
predictions_log   = model.predict(test)
predictions_final = np.expm1(predictions_log)

predictions_final = np.clip(predictions_final, a_min=10000, a_max=None)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': predictions_final})
submission.to_csv('submission.csv', index=False)

print(submission['SalePrice'].describe())
print(f"\nSubmission saved: {len(submission)} rows")
print(f"Sample predictions:\n{submission.head(10).to_string(index=False)}")

count      1459.000000
mean     178540.332517
std       77626.040675
min       43588.059053
25%      127721.396728
50%      156204.999962
75%      207487.874952
max      557067.462481
Name: SalePrice, dtype: float64

Submission saved: 1459 rows
Sample predictions:
  Id     SalePrice
1461 121877.732411
1462 164642.254021
1463 178839.535171
1464 196659.662136
1465 183359.258439
1466 172821.752978
1467 176772.875395
1468 170618.991494
1469 184557.007540
1470 125891.647777
